# Retrieval → reranker stage smoke test

Runs the normal cached multi-dense retrieval over the full deterministic dev split, replaces the retrieval process image, then reranks only 20 queries. This notebook is diagnostic and must not be used for reported metrics or LightGBM training.

In [ ]:
from pathlib import Path
import os, subprocess, yaml

REPO=Path('/kaggle/working/Drill-reliable-retrieval')
if not REPO.is_dir():
    subprocess.run(['git','clone','https://github.com/lmnhnht125/Drill-reliable-retrieval.git',str(REPO)],check=True)
subprocess.run(['python','-m','pip','install','-e',str(REPO)],check=True)
os.chdir(REPO)


In [ ]:
def one_dir(name):
    matches=[p for p in Path('/kaggle/input').rglob(name) if p.is_dir()]
    if len(matches) != 1: raise RuntimeError(f'Expected one {name} cache, found: {matches}')
    return matches[0]
def one_file(name):
    matches=list(Path('/kaggle/input').rglob(name))
    if len(matches) != 1: raise RuntimeError(f'Expected one {name}, found: {matches}')
    return matches[0]

CORPUS=one_file('legal-corpus.parquet'); TRAIN=one_file('train.parquet')
cfg=yaml.safe_load(Path('configs/hybrid_reranker_multi_dense.yaml').read_text())
cfg['experiment_name']='reranker_stage_smoke_20q'
cfg['retrieval']['bm25']['cache'].update(directory=str(one_dir('bm25s_vncorenlp')),require_hit=True)
cfg['retrieval']['dense']['chunk_cache'].update(directory=str(one_dir('bge_m3_khoan_diem448_chunks')))
cfg['retrieval']['dense_profiles']['vietlegal_harrier']['cache'].update(directory=str(one_dir('vietlegal_harrier_0_6b')),require_hit=True)
cfg['retrieval']['dense_profiles']['multilingual_e5_large']['cache'].update(directory=str(one_dir('multilingual_e5_large')),require_hit=True)
cfg['reranking'].update(batching='per_query',batch_size=8,smoke_query_limit=20)
OUT=Path('/kaggle/working/reranker_stage_smoke'); OUT.mkdir(parents=True,exist_ok=True)
CONFIG=OUT/'smoke.yaml'; CONFIG.write_text(yaml.safe_dump(cfg,allow_unicode=True,sort_keys=False))
os.environ['DRILL_CUDA_TRACE_PATH']=str(OUT/'cuda_process_trace.jsonl')
print({'corpus':str(CORPUS),'train':str(TRAIN),'output':str(OUT)})


In [ ]:
subprocess.run([
    'python','scripts/run_experiment.py','--config',str(CONFIG),
    '--corpus-path',str(CORPUS),'--train-path',str(TRAIN),
    '--output-root',str(OUT),'--devices','2'
],check=True)


In [ ]:
run=OUT/cfg['experiment_name']
print('Smoke run completed:',run)
print((run/'metrics.json').read_text())
print('Do not use these 20-query metrics as research results.')
